### The purpose of this is to take people involved in movies and turn them into scores

The key idea is by looking at the median and average revenue of the movies people are in you can turn that into a score for them. There are two types of people those that are actors and those involved in production. Then the data is pivoted such that instead of multiple copies of the movie, there are just scores for production and actors in them, representing their starpower.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("movie-data/combined_df.csv")


In [4]:
# Movies contains all rows where the revenue is greater than 0.
movies_df = df[df["revenue"] > 0]
movies_df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,tagline,genres,production_companies,production_countries,spoken_languages,keywords,tconst,nconst,category,primaryName
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0000138,actor,Leonardo DiCaprio
1,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0330687,actor,Joseph Gordon-Levitt
2,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0680983,actor,Elliot Page
3,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0913822,actor,Ken Watanabe
4,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0362766,actor,Tom Hardy


In [ ]:
# A dataframe of rows only containing cast.
actor_roles = ['actor', 'actress', "self"]
actors_df = movies_df[movies_df['category'].isin(actor_roles)].copy()
# print(actors_df.head(20))

# A dataframe of rows only containing crew.
production_roles = ['director', 'producer', 'writer']
production_df = movies_df[movies_df['category'].isin(production_roles)].copy()
# print(production_df.head(20))

print("Actors/Actresses:", actors_df.shape)
print("Production:", production_df.shape)

Actors/Actresses: (164265, 25)
Production: (85897, 25)


In [10]:
# Compute profit per movie and put it in a new column.
actors_df['profit'] = actors_df['revenue'] - actors_df['budget']

# Aggregate profit data per actor
# Creates new dataframe where each row is an actor and the columns are 
# average profit, median profit, and number of titles they did.
agg_actors = actors_df.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

# The average of each actor's average profit and median profit in a column called Composite.
agg_actors['composite'] = (agg_actors['avg_profit'] + agg_actors['med_profit']) / 2

# Punish low number of titles: scale weight if num_titles < 4; otherwise weight=1
# Creates a weight column which is equal to 1 if an actor has been in 4 or more movies
# and is num_movies / 4 if they have been in less than 4.
agg_actors['weight'] = agg_actors['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

# Multiplies the composite by the weight.
agg_actors['final_composite'] = agg_actors['composite'] * agg_actors['weight']

# some statistics on the actor's scores.
mean_comp = agg_actors['final_composite'].mean()
std_dev_comp = agg_actors['final_composite'].std()
agg_actors['z_score'] = (agg_actors['final_composite'] - mean_comp) / std_dev_comp

print(mean_comp)
print(std_dev_comp)

# Transform z-score to final score; allow negative values for flops
agg_actors['final_score'] = 100 * agg_actors['z_score']

pd.set_option("display.float_format", "{:.2f}".format)

agg_actors.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head(20)


7084112.433064383
25268956.362306334


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
48650,nm1569276,Chadwick Boseman,11,838737521.13,905046416.00,871891968.57,1.00,871891968.57,34.22,3422.41
51790,nm1853544,Pierre Coffin,4,773256919.00,903090397.50,838173658.25,1.00,838173658.25,32.89,3288.97
317,nm0000355,Anthony Daniels,9,640437861.67,737000000.00,688718930.83,1.00,688718930.83,26.98,2697.52
37629,nm1019674,Sala Baker,4,528959764.17,778368364.00,653664064.08,1.00,653664064.08,25.59,2558.79
26101,nm0641063,Dean O'Gorman,4,546592665.75,707209894.00,626901279.88,1.00,626901279.88,24.53,2452.88
60867,nm3269138,Dana Gaier,3,770331315.00,894761885.00,832546600.00,0.75,624409950.00,24.43,2443.02
45672,nm1388927,Miranda Cosgrove,3,770331315.00,894761885.00,832546600.00,0.75,624409950.00,24.43,2443.02
59909,nm3094377,Willow Shields,4,619547860.50,624875717.50,622211789.00,1.00,622211789.00,24.34,2434.32
36107,nm0942247,Bonnie Wright,4,533989119.25,694132532.50,614060825.88,1.00,614060825.88,24.02,2402.06
63904,nm3918035,Zendaya,6,686488344.83,527005800.50,606747072.67,1.00,606747072.67,23.73,2373.12


In [11]:
print("Aggregated Actors/Actresses:")
agg_actors.sort_values('final_score', ascending=False)[['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 'weight', 'final_score']].head(15)

Aggregated Actors/Actresses:


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_score
48650,nm1569276,Chadwick Boseman,11,838737521.13,905046416.00,871891968.57,1.00,3422.41
51790,nm1853544,Pierre Coffin,4,773256919.00,903090397.50,838173658.25,1.00,3288.97
317,nm0000355,Anthony Daniels,9,640437861.67,737000000.00,688718930.83,1.00,2697.52
37629,nm1019674,Sala Baker,4,528959764.17,778368364.00,653664064.08,1.00,2558.79
26101,nm0641063,Dean O'Gorman,4,546592665.75,707209894.00,626901279.88,1.00,2452.88
60867,nm3269138,Dana Gaier,3,770331315.00,894761885.00,832546600.00,0.75,2443.02
45672,nm1388927,Miranda Cosgrove,3,770331315.00,894761885.00,832546600.00,0.75,2443.02
59909,nm3094377,Willow Shields,4,619547860.50,624875717.50,622211789.00,1.00,2434.32
36107,nm0942247,Bonnie Wright,4,533989119.25,694132532.50,614060825.88,1.00,2402.06
63904,nm3918035,Zendaya,6,686488344.83,527005800.50,606747072.67,1.00,2373.12


In [12]:
print(agg_actors[agg_actors["primaryName"] == "Brad Pitt"])

       nconst primaryName   avg_profit  med_profit  num_titles    composite  \
73  nm0000093   Brad Pitt 115512306.36 90845033.00          44 103178669.68   

    weight  final_composite  z_score  final_score  
73    1.00     103178669.68     3.80       380.29  


In [13]:
# Compute profit per movie and put it in a new column.
production_df['profit'] = production_df['revenue'] - production_df['budget']

# Aggregate profit data per crew member (production person)
# Creates new dataframe where each row is an actor and the columns are 
# average profit, median profit, and number of titles they did.
agg_production = production_df.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

# The average of each crew member's average profit and median profit in a column called Composite.
agg_production['composite'] = (agg_production['avg_profit'] + agg_production['med_profit']) / 2

# Apply a weight that scales if num_titles < 4; otherwise, weight = 1
# Creates a weight column which is equal to 1 if an crew member has been in 4 or more movies
# and is num_movies / 4 if they have been in less than 4.
agg_production['weight'] = agg_production['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

# Multiplies the composite by the weight.
agg_production['final_composite'] = agg_production['composite'] * agg_production['weight']

# some statistics on the crew's scores.
mean_comp = agg_production['final_composite'].mean()
std_comp = agg_production['final_composite'].std()
agg_production['z_score'] = (agg_production['final_composite'] - mean_comp) / std_comp

# Transform z-score to final score; allow negative values for flops
agg_production['final_score'] = 100 * agg_production['z_score']

agg_production.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head()


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
9501,nm0484457,Jon Landau,6,1138348493.50,1047615412.00,1092981952.75,1.00,1092981952.75,27.60,2759.74
23243,nm1601644,Jennifer Lee,4,864065106.17,1124219009.00,994142057.58,1.00,994142057.58,25.08,2507.91
3010,nm0118333,Chris Buck,4,756396104.14,1124219009.00,940307556.57,1.00,940307556.57,23.71,2370.75
24507,nm1853544,Pierre Coffin,4,848431226.75,923157235.00,885794230.88,1.00,885794230.88,22.32,2231.86
20536,nm1273099,Erik Sommers,5,862747749.40,905339117.00,884043433.20,1.00,884043433.20,22.27,2227.40


### Ok now here you is where you add the columns you just calculated

In [ ]:
# This adds the new actor final scores into the movies dataframe.
# So each row is a single movie and a single actor with their score. (hasn't been pivoted)
movies_with_actor_scores_df = pd.merge(movies_df, agg_actors[['nconst', 'final_score']],
                                    on='nconst', how='left', suffixes=('', '_actor'))

# This compiles statistics on each movie. So for each movie it calculates
# the average of all the actors' scores, their median, and standard deviation.
actor_stats_df = movies_with_actor_scores_df[movies_with_actor_scores_df['category'].isin(actor_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(actor_avg='mean', actor_med='median', actor_dev='std') \
    .reset_index()

# Repeats the process above for the crew.
movies_with_prod_scores_df = pd.merge(movies_df, agg_production[['nconst', 'final_score']],
                                   on='nconst', how='left', suffixes=('', '_prod'))

# Repeats the process above for the crew.
prod_stats_df = movies_with_prod_scores_df[movies_with_prod_scores_df['category'].isin(production_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(production_avg='mean', production_med='median', production_dev='std') \
    .reset_index()

# Filters out everything but director, producer, and writer.
prod_only_df = movies_with_prod_scores_df[movies_with_prod_scores_df['category'].isin(production_roles)]

#prod_stats_df.head(10)
#actor_stats_df.head(10)
#movies_with_actor_scores_df[["title", "category", "primaryName", "final_score"]].head(10)

prod_only_df[["title", "category", "primaryName", "final_score"]].head(10)


       tconst  actor_avg  actor_med  actor_dev
0   tt0003471     -22.74     -24.50       4.46
1   tt0004391     -23.13     -24.33       3.36
2   tt0004545     -27.68     -27.88       0.72
3   tt0004630     -22.15     -23.98       3.49
4   tt0004972     -17.00     -17.21       2.33
5   tt0005078     -23.15     -27.92      10.50
6   tt0005393     -27.88     -27.88       0.06
7   tt0006140     -25.27     -27.87       8.21
8   tt0006333     -19.54     -20.32       2.35
9   tt0006568     -25.98     -27.12       2.01
10  tt0006864     -20.81     -22.77       5.98
11  tt0007038     -26.33     -27.87       4.04
12  tt0009105     -28.03     -28.03       0.00
13  tt0009369     -20.12     -20.37       1.93
14  tt0009652     -24.07     -25.66       2.79
15  tt0009968     -23.35     -25.66       4.64
16  tt0010040     -22.74     -23.19       1.31
17  tt0010178     -27.09     -27.88       1.80
18  tt0010323     -27.42     -28.04       2.55
19  tt0010466     -24.41     -24.59       0.44


In [ ]:
movies_unique = movies_df.drop_duplicates(subset=['tconst']).copy()

movies_final = movies_unique.merge(actor_stats, on='tconst', how='left') \
    .merge(prod_stats_df, on='tconst', how='left')
movies_final = movies_final.drop(columns=['nconst', "tconst", "primaryName", "category"])
movies_final.head()


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,production_companies,production_countries,spoken_languages,keywords,actor_avg,actor_med,actor_dev,production_avg,production_med,production_dev
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",338.459844,246.267534,238.142839,1054.527486,1034.746057,39.562860
1,157336,Interstellar,8.417,32571,Released,11/5/2014,701729206,169,False,165000000,...,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",258.618166,154.114164,222.647814,862.125977,1034.746057,376.005072
2,155,The Dark Knight,8.512,30619,Released,7/16/2008,1004558444,152,False,185000000,...,"DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",267.919374,191.809402,210.527878,773.315771,836.477049,328.158370
3,19995,Avatar,7.573,29815,Released,12/15/2009,2923706026,162,False,237000000,...,"Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",625.067922,511.999718,322.760089,1620.716567,1241.073114,759.286907
4,24428,The Avengers,7.710,29166,Released,4/25/2012,1518815515,143,False,220000000,...,Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",883.491854,865.975453,283.026839,750.170603,566.326504,629.076898


In [126]:
movies_final.to_csv("movie-data/movies_with_scores.csv", index=False)
print("Saved movies_with_scores.csv")

Saved movies_with_scores.csv


In [127]:
print("Top 10 movies with highest actor average score:")
top_actor_avg = movies_final.sort_values('actor_avg', ascending=False).head(10)
print(top_actor_avg[['title', 'actor_avg', 'actor_med', 'actor_dev']])
print("Top 10 movies with highest production average score:")  
top_prod_avg = movies_final.sort_values('production_avg', ascending=False).head(10)
print(top_prod_avg[['title', 'production_avg', 'production_med', 'production_dev']])

Top 10 movies with highest actor average score:
                                                 title    actor_avg  \
131                                      Despicable Me  1599.311881   
27                                       Black Panther  1433.743253   
550                                    Despicable Me 3  1418.694339   
6                               Avengers: Infinity War  1390.439711   
71                   The Hobbit: An Unexpected Journey  1285.659387   
19   The Lord of the Rings: The Fellowship of the Ring  1211.470517   
15                                   Avengers: Endgame  1209.304141   
65        Harry Potter and the Deathly Hallows: Part 1  1194.981458   
129                           Star Wars: The Last Jedi  1190.508317   
179                The Hobbit: The Desolation of Smaug  1072.493614   

       actor_med    actor_dev  
131  1013.769427  1250.230198  
27   1318.442031  1142.255621  
550   800.588075  1232.272741  
6     887.291056   905.480310  
71    929.